<img src="https://datasciencedegree.wisconsin.edu/wp-content/themes/data-gulp/images/logo.svg" width="300">
<img src="https://pandas.pydata.org/static/img/pandas.svg" width="200">

 # <center>Notebook - Lesson 6a - Series, Frame, Element </center>

This notebook aims to help you understand what kind of thing you have after taking an action when using the pandas data framing library in Python.

I'm writing this notebook because I left myself the following notes from an earlier instance of this class:

* Because idxmax is used a lot in Assignment 6a, teach directly about indices, `loc`, and `iloc`.
* You need to be able to predict after a pandas operation whether you have a `DataFrame`, a `Series`, or a single entry.
* When selecting data using multiple conditions, the way to join filters is `&`, not `and`, and that one must use parentheses around each condition.

---

## Imports up top

We'll start by importing NumPy and pandas:

In [ ]:
import numpy as np
import pandas as pd

We'll again use the `elements-by-episode.csv` data set.  Make sure you have it in the same directory as this notebook (Source: <a href="https://github.com/fivethirtyeight/data">FiveThirtyEight</a>). 


The data given indicates what Bob Ross painted in each episode of "The Joy of Painting", which aired new episodes on PBS from 1983 to 1994. A "1" in an entry indicates that Ross painted the item tagged in that episode, while a "0" indicates that the tagged object was not present in an episode. 

In [ ]:
bob_ross = pd.read_csv('elements-by-episode.csv')

---

## indices, `loc`, and `iloc`.

Pandas data frames are fancy wrappers around a rectangular data base.  Each row in a `DataFrame` has an "index" associated to it.  By default, the indices are non-negative integers just like lists, tuples, and numpy arrays.

You can get information on the current indices for a `DataFrame` using the `.index()` function:

In [ ]:
bob_ross.index

The min index is 0, the max is 402, there are a total of 403, and the step between them is `1`.  

### `iloc` -- get data at integer locations

If we want a particular row from a data set, we can get it using the `iloc` function.  It's a suprising one at first pass, since it uses square braces `[]`, not round braces, when you specify the indices you want.  

Suppose we want the row at position 100:

In [ ]:
bob_ross.iloc[100]

The `i` in `iloc` stands for `integer`.  The `iloc` function gets data from a pandas object with matching integer locations.  

We can also use `iloc` to get multiple elements:

In [ ]:
bob_ross.iloc[100:105]

### `loc`


There's another function to tell you about: `loc`.  This is another accessor for retrieving data from a pandas object, and stands in contrast to `iloc` in that it works with non-integer data, particularly for the scenario when we index a data set not by arbitrary rows-as-read-in, but perhaps by a more meaningful column with unique values (so that no two rows have the same index -- otherwise it's not a good index!)

Let's make a new data frame by setting the index to the the episode title -- a good choice, since no two episodes have the same title.  (Another good choice would be episode number...)

In [ ]:
bob_ross_indexed_by_title = bob_ross.set_index('TITLE')  # use `inplace=True` if you want to modify the existing data frame
# make new data frame, same as old but using a more "natural" index -- episode title

We can now use the `.loc` operator -- again with square braces, not round -- to get data from the data frame.  

For example,

In [ ]:
bob_ross_indexed_by_title.loc['"A WALK IN THE WOODS"']

Note that the double-quotes in the titles are super annoying.  Here's some code to remove them, so that we never have to type them again:

In [ ]:
bob_ross['TITLE'] = bob_ross['TITLE'].str.strip('"') # remove the double quotes from the original data set
bob_ross_indexed_by_title = bob_ross.set_index('TITLE') # overwrite indexed-by-title, now have no ""

Now we can use `.loc` to get episodes without the stupid double quotes messing everything up:

In [ ]:
bob_ross_indexed_by_title.loc['A WALK IN THE WOODS'] # yay, no extra quotes

### `iloc` on objects *not* using default indices

Now that `bob_ross_indexed_by_title` has non-default non-integer indices, I hope you're curious about the behaviour of `iloc` on this data frame.  Let's check it out!

Using `.iloc` on an alternatively-indexed data frame still works:

In [ ]:
bob_ross_indexed_by_title.iloc[0]

### Slicing and `iloc`

💡 What if we select just part of a non-default-indexed data frame, and use `iloc` on that frame?  

Using `bob_ross_indexed_by_title`, let's grab the second season only:

In [ ]:
# slice
season_2_indexed_by_title = bob_ross_indexed_by_title[bob_ross_indexed_by_title['EPISODE'].str.startswith('S02')]

season_2_indexed_by_title.head(3)

Now let's use `iloc[0]` on this slice and see what we get:

In [ ]:
season_2_indexed_by_title.iloc[0]

ℹ️ Outcome: using `iloc` on a DataFrame (or Series) uses an absolute location.  The first row in the data frame, slice or not, indexed using default indices or not, always is accessible via `.iloc[0]`

### Summary of iloc and loc

* Every `DataFrame` and `Series` in pandas has an index column associated to it.  By default, the indices are non-negative integers, and come in the order in which the data was imported or created.  

* You can change from non-negative integers to anything you wish using `reindex`, and the new index column should be unique for every row.  

* To use absolute 0-based integer locations, use `.iloc[]`.  You can use `iloc` even if you indexed on a different column.  

* The first row in a data frame always is accessible via `.iloc[0]`  (assuming non-empty frame)

* To use non-integer indices, after re-indexing to use another column, use `.loc[]`.  


## Fundamental pattern to select data

The fundamental pattern you need to use is this:

```
df[df[colname]==something]
```

or more generally,

```
df[ logical statement using df]
```

These two patterns provide much functionality to select only the data you want or need from a series or data frame.  

Example selection statements

In [ ]:
has_a_boat = bob_ross[ bob_ross['BOAT']==1 ]
has_a_boat.head(5)

Let's get the first episode of each season:

In [ ]:
first_episodes_of_seasons = bob_ross[ bob_ross['EPISODE'].str.endswith('E01') ]
assert first_episodes_of_seasons.shape[0] == 31  # check that we have all the episodes

#### Understanding the sequence of events in the `df[conditional(df)]` pattern

What actually happens in the sequence of events when selecting data using the `df[conditional(df)]` pattern?  

1. Generate a true/false Series object using `conditional(df)`
2. Select data from data frame using that TF series

Let's check it out.  Turning back to the "has a boat" example, let's check out what the inside conditional statement generates:

In [ ]:
# generates a column of true/false values
bob_ross['BOAT']==1

In [ ]:
# trues are 1, falses are 0, so `sum` computes the number that matched.
sum(bob_ross['BOAT']==1)

Next, we use that TF series to actually select the data we want.  Only rows with a `True` get kept through the slice.

In [ ]:
bob_ross[ bob_ross['BOAT']==1 ]

### Selecting on multiple conditions

One skill to have in pandas is to be able to select data where multiple conditions are true, using the `df[conditional(df)]` pattern.  Pandas is suprising to newcomers, because chaining together the True/False series objects does NOT use the `and` and `or` keywords, but instead the `&` and `|` symbolic binary operators.

My test example here will be to select the paintings that have both a beach and a boat.  There's exactly one of them, so this will lead to my next point about accessing items when a slice has one thing in it.

First, we'll produce some boolean series objects that contain the indicators:

In [ ]:
has_a_boat = bob_ross['BOAT']==1
has_a_beach = bob_ross['BEACH']==1

Next, we'll try to use the `and` keyword to produce a TF series telling us if the corresponding painting has both:

In [ ]:
# this produces an error.  Don't use `and` for this operation.  
has_a_boat and has_a_beach

You should have gotten the following error message:

```
ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
```

Indeed, since a Series contains many values, simple logical `and` in Python doesn't work.  `and` wants a single true/false value on either side, but we have many.  

I wish the error message contained the correct change to make: use `&` instead of `and` to take element-wise.  Similarly, `|` instead of `or`.

Observe:

In [ ]:
has_both = has_a_boat & has_a_beach
sum(has_both)  # there's exactly one

Finally, we can use this boolean series to perform a slice using `[]` on the data frame.

In [ ]:
bob_ross[has_both]

Done in one line, as is the goal, we have:

In [ ]:
bob_ross_boat_and_beach = bob_ross[has_a_boat & has_a_beach]

## When selecting data using multiple conditions, the way to join filters is `&`, not `and`, and that one must use parentheses around each condition.

I have one more thing to teach you about multiple conditions when slicing: group the individual conditionals in parentheses `()`.

First, I'll write what I naively think *should* work:

In [ ]:
# I want this to work, but it doesn't!
bob_ross_boat_and_beach = bob_ross[bob_ross['BOAT']==1 & bob_ross['BEACH']==1]
# this generates an error

We get that same error again!!!!! Ugh.  The `&` happens *before* the `==` comparison!!!!  Yes, `&` is before `==`, whereas `and` is after.  See [the table of operator precedence in official Python documentation](https://docs.python.org/3/reference/expressions.html#operator-precedence).

Thus, we have to group the individual TF series using `()`, and put the `&` between them.  Observe:

In [ ]:
# this works because of the () around each individual condition
bob_ross_boat_and_beach = bob_ross[(bob_ross['BOAT']==1) & (bob_ross['BEACH']==1)]
#                                  ^                   ^   ^                    ^
#                              added ()                 & between             added ()

---

## How to predict whether I have a `DataFrame`, a `Series`, or a single entry

When you take an action in pandas, a critical skill is to reliably predict the type of the result.  Generally, for the operations I'm currently training you on, you'll get one of these upon an action:

* `DataFrame` --- basically, a rectangular data set
* `Series` --- basically, a column of data
* element or item

In [ ]:
# a helper function.
print_type = lambda x: print('`{}` is of type {}'.format(x, type(eval(x))))
#                                                               ^ 
#                                       the first appearance of `eval` in the course.
#                                       ⚠️ use `eval` with extreme caution.
#
# lets me put a string of code and print and run it at the same time.

## Slicing a data frame using `df[condition(df)]` gives another `DataFrame`

In [ ]:
print_type("bob_ross[bob_ross['EPISODE'].str.startswith('S24')]")

## Selecting a single column from a data frame gives a `Series`

In [ ]:
season24 = bob_ross[bob_ross['EPISODE'].str.startswith('S24')]

print_type("season24['TITLE']")

## Selecting using a single value in `.iloc` and `.loc` give a `Series`

In [ ]:
print_type('season24.iloc[0]')

## Selecting using multiple values in `.loc` or `.loc` give a `DataFrame`

In [ ]:
print_type('season24.iloc[0:3]')

## When you want a single value

This last section discusses a pattern for getting values from a DataFrame, when you know you want only one value from the frame, and you're selecting to get the value.  

A primary context for this might be "find the title for the episode with the most elements".

First, we'll construct a list of the column names for the painting elements.  We exclude title, episode number, and guest painter information, and information about what type of frame was used:

In [ ]:
# a list comprehension with two conditionals to filter.
element_names = [c for c in bob_ross.columns if (c not in ['EPISODE','TITLE','DIANE_ANDRE','GUEST']) and ('FRAME' not in c)]

# the first three
element_names[:3]

Next we'll select the columns we want.

In [ ]:
painting_elements = bob_ross[element_names]
painting_elements.head(3)

Now I want the total number used in any painting.  That's a row-sum, so I'll use the `sum` operator with the `axis=1` argument, and add it to the full data frame:

In [ ]:
bob_ross['num_elements'] = painting_elements.sum(axis=1)

Now we can find the maximum number of elements in any one painting:

In [ ]:
bob_ross['num_elements'].max()

How many paintings had 14 things?

In [ ]:
has_max_elements = bob_ross[ bob_ross['num_elements']==bob_ross['num_elements'].max() ]
print('there are {} paintings with the max number of elements'.format(has_max_elements.shape[0]))

Let's get the title of that episode.  We'll use the `.iloc` function, since it uses absolute integer indices, and we want the 0th thing:

In [ ]:
has_max_elements.iloc[0]['TITLE']

All in one line:

In [ ]:
bob_ross[ bob_ross['num_elements']==bob_ross['num_elements'].max() ].iloc[0]['TITLE']

A little wordy though, don't you think?  The next subsection tells you a faster way, using `idxmax`.

---

There's another way to get this, that I think is simpler.  We can use the `idxmax` function, which tells us the index of the row in which the max occurred.  Check it out:

In [ ]:
index_of_max_elements = bob_ross['num_elements'].idxmax()

And then we can use the returned index to get the painting (row).  We'll use `loc`, not `iloc`, since this is an index and not an absolute location.

In [ ]:
bob_ross.loc[index_of_max_elements]

All in one line!!!  

In [ ]:
bob_ross.loc[ bob_ross['num_elements'].idxmax()]['TITLE']

So nice!  The `idxmax` function helps us get the data we want. In this case, we wanted the title of the episode with the maximum number of elements, NOT the max number of elements.

Advice: If you use a column to find the data you want, and you want the value from a different column, `idxmax` is probably for you.  

There are also other `.idxFN` functions, where `FN` is a placeholder for whatever operation you want, in particuar `min`.

---

Credits: Written by silviana amethyst, phd.  spring 2023, fall 2023.